In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import glob
from xgrads import open_CtlDataset
from pathlib import Path
import netCDF4

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output

import matplotlib.colors as mcolors

import ipywidgets as widgets
from IPython.display import display, clear_output

ncl_cmap = LinearSegmentedColormap.from_list(
    "BlueWhiteOrangeRed",
    ["#2166ac", "#67a9cf", "#ffffff", "#fdae61", "#b2182b"],
    N=256
)


plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_colwidth", 160)

print("Python OK")

Python OK


In [6]:
from pathlib import Path

BASE_DIR = Path.cwd()

NEW_DIR = BASE_DIR / ".." / ".." / ".." /"SPEEDY_access" / "output" / "exp_102"
OLD_DIR = BASE_DIR / ".." / ".." / ".." /"SPEEDY_access" / "output" / "exp_101"

print("NEW_DIR:", NEW_DIR, NEW_DIR.exists())
print("OLD_DIR:", OLD_DIR, OLD_DIR.exists())

# SPEEDY -> ACCESS-OM2 variable names
SPEEDY_VARIABLES = {
    "SLR": "rlds",
    "SSR": "rsds",
}

# Output directory
FORCING_OUT_DIR = (BASE_DIR / ".." / ".." / ".." / "SPEEDY_access" / "access_forcing").resolve()
FORCING_OUT_DIR.mkdir(parents=True, exist_ok=True)

WRITE_ONE_FILE_PER_YEAR = True

# Keep SPEEDY grid untouched until JRA-55 metadata/grid are inspected
SHIFT_LONGITUDE_TO_MINUS180_180 = False
SORT_LATITUDE_NORTH_TO_SOUTH = False

print("Output directory:", FORCING_OUT_DIR)
print("Variables:", SPEEDY_VARIABLES)

NEW_DIR: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/scripts/access_forcing/../../../SPEEDY_access/output/exp_102 True
OLD_DIR: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/scripts/access_forcing/../../../SPEEDY_access/output/exp_101 True
Output directory: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing
Variables: {'SLR': 'rlds', 'SSR': 'rsds'}


In [7]:
for ctl in Path(NEW_DIR).glob("attm102.ctl"):
    print("=" * 80)
    print(ctl.name)

    ds = open_CtlDataset(str(ctl))
ds

attm102.ctl


<xarray.Dataset> Size: 8GB
Dimensions:  (time: 4380, lev: 8, lat: 48, lon: 96)
Coordinates:
  * time     (time) datetime64[ns] 35kB 1989-01-01 ... 1991-12-31T18:00:00
  * lev      (lev) float64 64B 925.0 850.0 700.0 500.0 300.0 200.0 100.0 30.0
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
Data variables: (12/39)
    GH       (time, lev, lat, lon) >f4 646MB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    TEMP     (time, lev, lat, lon) >f4 646MB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    U        (time, lev, lat, lon) >f4 646MB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    V        (time, lev, lat, lon) >f4 646MB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    Q        (time, lev, lat, lon) >f4 646MB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    RH       (time, lev, lat, lon) >f4 646MB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    ...       ...
    OLR      (time, lat, lon) >f4 81MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SSR      (time, lat, lon) >f4 81MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SLR      (time, lat, lon) >f4 81MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SHF      (time, lat, lon) >f4 81MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    LSHF     (time, lat, lon) >f4 81MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SSHF     (time, lat, lon) >f4 81MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
Attributes:
    comment:  geopotential height               [m]
    storage:  99
    title:    Means/variances
    undef:    9.999e+19
    pdef:     None

In [8]:
# Check SPEEDY variables

for speedy_var, access_var in SPEEDY_VARIABLES.items():

    if speedy_var not in ds:
        raise KeyError(
            f"{speedy_var!r} is absent from the SPEEDY dataset. "
            f"Available variables: {list(ds.data_vars)}"
        )

    var = ds[speedy_var]

    print(f"\n{'='*60}")
    print(f"{speedy_var} -> {access_var}")
    print(f"{'='*60}")

    print(var)
    print("Dimensions:", var.dims)
    print("Shape:", var.shape)
    print("Dtype:", var.dtype)
    print("Attributes:", var.attrs)

    print(
        f"{speedy_var} range:",
        float(var.min().compute()),
        "to",
        float(var.max().compute()),
    )

    print(
        f"{speedy_var} global mean:",
        float(var.mean(skipna=True).compute()),
    )

# Check temporal resolution only once
dt_hours = np.diff(ds.time.values) / np.timedelta64(1, "h")
print("\nUnique output intervals [hours]:", np.unique(dt_hours))


SLR -> rlds
<xarray.DataArray 'SLR' (time: 4380, lat: 48, lon: 96)> Size: 81MB
dask.array<reshape, shape=(4380, 48, 96), dtype=>f4, chunksize=(1, 48, 96), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 35kB 1989-01-01 ... 1991-12-31T18:00:00
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
Attributes:
    comment:  surface longwave rad.
    storage:  99
Dimensions: ('time', 'lat', 'lon')
Shape: (4380, 48, 96)
Dtype: >f4
Attributes: {'comment': 'surface longwave rad.', 'storage': '99'}
SLR range: -213.3104705810547 to 406.06549072265625
SLR global mean: 73.93325805664062

SSR -> rsds
<xarray.DataArray 'SSR' (time: 4380, lat: 48, lon: 96)> Size: 81MB
dask.array<reshape, shape=(4380, 48, 96), dtype=>f4, chunksize=(1, 48, 96), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 35kB 1989-01-01 ... 1991-12-31T18:00:00
  * lat      (lat) flo

In [9]:
# Build ACCESS-OM2 forcing variables

forcing_vars = {}

for speedy_var, access_var in SPEEDY_VARIABLES.items():

    var = ds[speedy_var].rename(access_var).astype("float32")

    # Unit conversion
    # if speedy_var == "MSLP":
    #     var = var * 100.0   # hPa -> Pa

    # Optional grid transformations
    if SHIFT_LONGITUDE_TO_MINUS180_180:
        var = var.assign_coords(
            lon=(((var.lon + 180.0) % 360.0) - 180.0)
        ).sortby("lon")

    if SORT_LATITUDE_NORTH_TO_SOUTH:
        var = var.sortby("lat", ascending=False)

    # Variable metadata
    if access_var == "rlds":
    var.attrs = {
        "standard_name": "surface_downwelling_longwave_flux_in_air",
        "long_name": "Surface downwelling longwave radiation",
        "units": "W m-2",
        "source_variable": "SLP",
        "source_model": "SPEEDY",
        "mapping_note": "Provisional mapping SPEEDY SLP to ACCESS-OM2 rlds",
    }

    elif access_var == "rsds":
    var.attrs = {
        "standard_name": "surface_downwelling_shortwave_flux_in_air",
        "long_name": "Surface downwelling shortwave radiation",
        "units": "W m-2",
        "source_variable": "SSR",
        "source_model": "SPEEDY",
        "mapping_note": "Provisional mapping SPEEDY SSR to ACCESS-OM2 rsds",
    }

forcing_vars[access_var] = var


# Build Dataset
forcing_ds = xr.Dataset(forcing_vars)


# Coordinate metadata
if "lat" in forcing_ds.coords:
    forcing_ds["lat"].attrs.update({
        "standard_name": "latitude",
        "long_name": "latitude",
        "units": "degrees_north",
        "axis": "Y",
    })

if "lon" in forcing_ds.coords:
    forcing_ds["lon"].attrs.update({
        "standard_name": "longitude",
        "long_name": "longitude",
        "units": "degrees_east",
        "axis": "X",
    })

forcing_ds["time"].attrs.update({
    "standard_name": "time",
    "long_name": "time",
    "axis": "T",
})


# Global metadata
forcing_ds.attrs = {
    "title": "SPEEDY forcing for ACCESS-OM2",
    "source": "SPEEDY model output",
    "history": "Created from SPEEDY SSR and SLR",
    "comment": (
        "Provisional ACCESS-OM2 forcing export. "
        "SPEEDY SSR -> rsds, SLR -> rlds. "
    ),
}

forcing_ds

IndentationError: expected an indented block after 'if' statement on line 23 (876077306.py, line 24)

In [9]:
required_dims = {"time", "lat", "lon"}

expected_units = {
    "uas": "m s-1",
    "vas": "m s-1",
    "psl": "Pa",
}

for var_name in ["uas", "vas", "psl"]:
    var = forcing_ds[var_name]

    if set(var.dims) != required_dims:
        raise ValueError(
            f"Expected {var_name} dimensions {required_dims}, got {var.dims}"
        )

    if var.attrs.get("units") != expected_units[var_name]:
        raise ValueError(
            f"{var_name} must have units {expected_units[var_name]!r}, "
            f"got {var.attrs.get('units')!r}"
        )

    if not np.issubdtype(var.dtype, np.floating):
        raise TypeError(f"{var_name} must be floating point, got {var.dtype}")

    # Check for NaN / inf
    invalid_count = int((~np.isfinite(var)).sum().compute())

    if invalid_count:
        raise ValueError(
            f"Found {invalid_count} invalid values in {var_name}"
        )

    var_min = float(var.min(skipna=True).compute())
    var_max = float(var.max(skipna=True).compute())

    # Broad sanity checks
    if var_name in ("uas", "vas"):
        if var_min < -150.0 or var_max > 150.0:
            print(
                f"WARNING: unusual {var_name} range: "
                f"{var_min:.3f} to {var_max:.3f} m s-1"
            )

    elif var_name == "psl":
        if var_min < 80000.0 or var_max > 110000.0:
            print(
                f"WARNING: unusual psl range: "
                f"{var_min:.1f} to {var_max:.1f} Pa"
            )

    print(f"{var_name}: {var_min:.3f} to {var_max:.3f} {expected_units[var_name]}")


print("\nValidation passed")
print("time:", forcing_ds.time.values[0], "to", forcing_ds.time.values[-1])
print("grid:", forcing_ds.sizes["lat"], "x", forcing_ds.sizes["lon"])

uas: -40.472 to 38.587 m s-1
vas: -43.927 to 42.949 m s-1
psl: 94387.898 to 105978.781 Pa

Validation passed
time: 1989-01-01T00:00:00.000000000 to 1991-12-31T18:00:00.000000000
grid: 48 x 96


In [10]:
field_encoding = {
    "dtype": "float32",
    "zlib": True,
    "complevel": 4,
    "shuffle": True,
    "_FillValue": np.float32(1.0e20),
    "chunksizes": (1, forcing_ds.sizes["lat"], forcing_ds.sizes["lon"]),
}

encoding = {
    **{v: field_encoding.copy() for v in ["uas", "vas", "psl"]},
    "time": {
        "units": "hours since 1900-01-01 00:00:00",
        "calendar": "365_day",
    },
    "lat": {"dtype": "float64", "_FillValue": None},
    "lon": {"dtype": "float32", "_FillValue": None},
}

In [11]:
written_files = []

if WRITE_ONE_FILE_PER_YEAR:
    years = np.unique(forcing_ds.time.dt.year.values)

    for year in years:
        yearly = forcing_ds.sel(time=str(int(year)))

        for var_name in ["uas", "vas", "psl"]:
            var_ds = yearly[[var_name]]
            output_file = FORCING_OUT_DIR / f"{var_name}_SPEEDY_{int(year)}.nc"

            # Encoding only for variables actually present in this file
            var_encoding = {
                var_name: encoding[var_name],
                "time": encoding["time"],
                "lat": encoding["lat"],
                "lon": encoding["lon"],
            }

            var_ds.to_netcdf(
                output_file,
                mode="w",
                format="NETCDF4",
                engine="netcdf4",
                unlimited_dims=["time"],
                encoding=var_encoding,
            )

            written_files.append(output_file)

            print(
                f"Wrote {output_file.name}: "
                f"{var_ds.sizes['time']} records, "
                f"{output_file.stat().st_size / 1024**2:.2f} MB"
            )

else:
    for var_name in ["uas", "vas", "psl"]:
        var_ds = forcing_ds[[var_name]]
        output_file = FORCING_OUT_DIR / f"{var_name}_SPEEDY_all_years.nc"

        var_encoding = {
            var_name: encoding[var_name],
            "time": encoding["time"],
            "lat": encoding["lat"],
            "lon": encoding["lon"],
        }

        var_ds.to_netcdf(
            output_file,
            mode="w",
            format="NETCDF4",
            engine="netcdf4",
            unlimited_dims=["time"],
            encoding=var_encoding,
        )

        written_files.append(output_file)

        print(
            f"Wrote {output_file.name}: "
            f"{var_ds.sizes['time']} records, "
            f"{output_file.stat().st_size / 1024**2:.2f} MB"
        )

written_files

Wrote uas_SPEEDY_1989.nc: 1460 records, 21.59 MB
Wrote vas_SPEEDY_1989.nc: 1460 records, 21.96 MB
Wrote psl_SPEEDY_1989.nc: 1460 records, 14.84 MB
Wrote uas_SPEEDY_1990.nc: 1460 records, 21.59 MB
Wrote vas_SPEEDY_1990.nc: 1460 records, 21.94 MB
Wrote psl_SPEEDY_1990.nc: 1460 records, 14.86 MB
Wrote uas_SPEEDY_1991.nc: 1460 records, 21.59 MB
Wrote vas_SPEEDY_1991.nc: 1460 records, 21.93 MB
Wrote psl_SPEEDY_1991.nc: 1460 records, 14.86 MB


[PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/uas_SPEEDY_1989.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/vas_SPEEDY_1989.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/psl_SPEEDY_1989.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/uas_SPEEDY_1990.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/vas_SPEEDY_1990.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/psl_SPEEDY_1990.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/uas_SPEEDY_1991.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/vas_SPEEDY_1991.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/psl_SPEEDY_1991.nc')]

In [12]:
if not written_files:
    raise RuntimeError("No NetCDF files were written")

for check_file in written_files:
    var_name = check_file.name.split("_")[0]

    with xr.open_dataset(check_file, decode_times=True) as check:
        print(f"\n{'='*60}")
        print(check_file.name)
        print(f"{'='*60}")

        print(check)
        print("\nVariable attributes:")
        print(check[var_name].attrs)

        print("\nEncoding:")
        print(check[var_name].encoding)

        print("\nTime:")
        print(check.time.values[0], "to", check.time.values[-1])

        var_min = float(check[var_name].min())
        var_max = float(check[var_name].max())
        units = check[var_name].attrs.get("units", "")

        print(f"\nRange [{units}]: {var_min:.6f} to {var_max:.6f}")

        # Compare first exported field against corresponding source field
        year = int(check.time.dt.year.values[0])
        source = forcing_ds[var_name].sel(time=str(year))
        source_first = source.isel(time=0).compute()
        output_first = check[var_name].isel(time=0).load()

        max_abs_difference = float(
            np.abs(source_first - output_first).max()
        )

        print(
            "Maximum absolute difference after NetCDF round trip:",
            max_abs_difference,
        )

        if max_abs_difference != 0.0:
            print(
                "NOTE: a tiny difference may arise from dtype or NetCDF encoding."
            )

print("\nAll output files verified")


uas_SPEEDY_1989.nc
<xarray.Dataset> Size: 27MB
Dimensions:  (time: 1460, lat: 48, lon: 96)
Coordinates:
  * time     (time) object 12kB 1989-01-01 00:00:00 ... 1989-12-31 18:00:00
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
Data variables:
    uas      (time, lat, lon) float32 27MB ...
Attributes:
    title:    SPEEDY forcing for ACCESS-OM2
    source:   SPEEDY model output
    history:  Created from SPEEDY U0, V0 and MSLP
    comment:  Provisional ACCESS-OM2 forcing export. SPEEDY U0 -> uas, V0 -> ...

Variable attributes:
{'standard_name': 'eastward_wind', 'long_name': 'Near-surface eastward wind', 'units': 'm s-1', 'source_variable': 'U0', 'source_model': 'SPEEDY', 'mapping_note': 'Provisional mapping SPEEDY U0 to ACCESS-OM2 uas'}

Encoding:
{'dtype': dtype('float32'), 'zlib': True, 'szip': False, 'zstd': False, 'bzip2': False, 'blosc': False, 'shuffle': True, 'complevel': 